# Luxembourg School Access Through the scalable architecture

This notebook reruns the Luxembourg school case through the draft Scalable General Distances Architecture. The data and computation are real: Luxembourg OSM, WorldPop-derived population points, OSM school amenities, candidate sites, Pandana routing, and the existing pipeline cache. The scalable layer provides the modular contracts, shared data registry, and interchangeable routing strategy boundary.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path(r"C:\local\GIT\Public-Infrastructure-Service-Access\Research-Sandbox\drafts\scalable")
scalable_SRC_ROOT = PROJECT_ROOT / "src"
GENERAL_PIPELINE_ROOT = Path(r"C:\local\GIT\Public-Infrastructure-Service-Access\Research-Sandbox\general_distances_per_country")
for path in [scalable_SRC_ROOT, GENERAL_PIPELINE_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from scalable_distances.artifacts.decorators import derived_artifact, source_artifact
from scalable_distances.artifacts.spec import ArtifactSpec
from scalable_distances.core.context import DataContext
from scalable_distances.data.registry import DatasetHandle, DatasetKey
from scalable_distances.data.schemas import DistanceMatrixSchema, PointSchema
from scalable_distances.routing.base import RouterCapabilities
from scalable_distances.storage.repository import Repository

from countries.luxembourg import CFG as REAL_LUX_CFG
from distance_pipeline.cache import CacheManager
from distance_pipeline.candidate_builder import build_candidate_sites
from distance_pipeline.distance_matrix import compute_distances_polars
from distance_pipeline.facilities import deduplicate_osm_amenities, load_facilities
from distance_pipeline.io import download_file
from distance_pipeline.network import build_pandana_network, load_osm_network_data
from distance_pipeline.population import worldpop_to_points
from distance_pipeline.settings import PipelineSettings
from distance_pipeline.snapping import snap_points_to_nodes
from distance_pipeline.source_tables import prepare_points_as_sources, prepare_points_as_targets
from distance_pipeline.viz import to_point_geometries
from run_pipeline import candidate_layer_for_role, concat_layers, layer_signature, with_prefixed_ids


## Runtime Setup

Default mode uses `aggregate_factor=10`, matching the compact Luxembourg run. Set `RECOMPUTE_FULL_RESOLUTION = True` to use the full-resolution population raster.


In [ ]:
class InMemoryRepository(Repository):
    def __init__(self, root: Path):
        super().__init__(root=root, default_format="memory")
        self.objects: dict[str, Any] = {}

    def artifact_uri(self, spec: ArtifactSpec, fingerprint: str) -> str:
        return f"memory://{spec.name}/{fingerprint}"

    def exists(self, uri: str | None) -> bool:
        return bool(uri) and uri in self.objects

    def read_dataset(self, key: DatasetKey, schema=None) -> DatasetHandle:
        if key.uri is None:
            raise ValueError("Memory reads need a URI")
        return DatasetHandle(key=key, data=self.objects[key.uri], schema=schema)

    def write_dataset(self, handle: DatasetHandle, spec: ArtifactSpec) -> None:
        if handle.key.uri is None:
            raise ValueError("Memory writes need a URI")
        self.objects[handle.key.uri] = handle.data


RECOMPUTE_FULL_RESOLUTION = False
real_arch_aggregate_factor = None if RECOMPUTE_FULL_RESOLUTION else 10
real_arch_amenities = ["school"]
real_arch_source_layers = ["amenities", "candidates"]
real_arch_destination_layers = ["population"]
real_arch_source_cache_values = layer_signature(
    source_layers=real_arch_source_layers,
    destination_layers=real_arch_destination_layers,
    amenity_values=real_arch_amenities,
    source_table=None,
    destination_table=None,
)

real_arch_settings = PipelineSettings(
    population_threshold=1,
    sample_fraction=1,
    max_points=None,
    max_total_dist=1000,
    candidate_grid_spacing_m=None,
    candidate_max_snap_dist_m=None,
    force_recompute=False,
    verbose=True,
    save_context_map=False,
    show_context_map=False,
)

real_arch_cache = CacheManager(
    REAL_LUX_CFG,
    force_recompute=real_arch_settings.force_recompute,
    verbose=real_arch_settings.verbose,
)
real_arch_ctx = DataContext(
    run_id="luxembourg_school_architecture",
    repository=InMemoryRepository(PROJECT_ROOT / "data" / "runs" / "luxembourg_school_architecture"),
)

{
    "country": REAL_LUX_CFG.iso3,
    "aggregate_factor": real_arch_aggregate_factor,
    "amenities": real_arch_amenities,
    "source_cache_values": real_arch_source_cache_values,
    "cache_dir": str(real_arch_cache.cache_dir),
}


## Architecture Adapters

The adapter layer is intentionally thin: real pipeline functions remain private implementation choices, while the notebook exposes stable scalable contracts.


In [ ]:
@dataclass(frozen=True)
class RealNetworkBundle:
    nodes: pd.DataFrame
    edges: pd.DataFrame
    network: object


class RealPandanaRouter:
    name = "pandana.real_pipeline"
    contract_version = "0.1"
    capabilities = RouterCapabilities(distance=True)

    def prepare(self, network: RealNetworkBundle, config: dict[str, Any] | None = None) -> "RealPandanaRouter":
        self.network = network.network
        return self

    def snap(self, points):
        return points

    def route_many(self, targets: pd.DataFrame, sources: pd.DataFrame, *, threshold_km: float, max_total_dist: float | None):
        return compute_distances_polars(
            targets=targets,
            sources=sources,
            distance_threshold_largest=threshold_km,
            network=self.network,
            max_total_dist=max_total_dist,
            verbose=real_arch_settings.verbose,
        )


def ensure_real_luxembourg_inputs(cfg=REAL_LUX_CFG) -> dict[str, Path]:
    if not cfg.PBF_PATH.exists():
        download_file(cfg.PBF_URL, cfg.PBF_PATH)
    if not cfg.WORLDPOP_PATH.exists():
        download_file(cfg.WORLDPOP_URL, cfg.WORLDPOP_PATH)
    return {"pbf": cfg.PBF_PATH, "worldpop": cfg.WORLDPOP_PATH}


## Decorated Computation Steps

In [ ]:
@source_artifact(name="real_luxembourg_inputs", format_role="json")
def real_luxembourg_inputs() -> dict[str, Path]:
    return ensure_real_luxembourg_inputs(REAL_LUX_CFG)


@source_artifact(name="real_luxembourg_network", format_role="network")
def real_luxembourg_network() -> RealNetworkBundle:
    nodes, edges = real_arch_cache.load_or_build_network_data(
        builder=lambda: load_osm_network_data(
            REAL_LUX_CFG.PBF_PATH,
            verbose=real_arch_settings.verbose,
            bbox=real_arch_settings.bbox,
        ),
        bbox=real_arch_settings.bbox,
    )
    return RealNetworkBundle(nodes=nodes, edges=edges, network=build_pandana_network(nodes, edges))


@source_artifact(name="real_luxembourg_population_points", schema=PointSchema, format_role="table")
def real_luxembourg_population_points() -> pd.DataFrame:
    return real_arch_cache.run(
        cache_path=real_arch_cache.population_points_path(
            population_threshold=real_arch_settings.population_threshold,
            sample_fraction=real_arch_settings.sample_fraction,
            max_points=real_arch_settings.max_points,
            aggregate_factor=real_arch_aggregate_factor,
        ),
        builder=lambda: worldpop_to_points(
            REAL_LUX_CFG.WORLDPOP_PATH,
            population_threshold=real_arch_settings.population_threshold,
            sample_fraction=real_arch_settings.sample_fraction,
            max_points=real_arch_settings.max_points,
            aggregate_factor=real_arch_aggregate_factor,
            verbose=real_arch_settings.verbose,
        ),
    )


@source_artifact(name="real_luxembourg_school_points", schema=PointSchema, format_role="table")
def real_luxembourg_school_points() -> pd.DataFrame:
    features = real_arch_cache.run(
        cache_path=real_arch_cache.facilities_path(amenity_values=real_arch_amenities),
        builder=lambda: load_facilities(
            REAL_LUX_CFG.PBF_PATH,
            amenity_values=real_arch_amenities,
            verbose=real_arch_settings.verbose,
        ),
    )
    return real_arch_cache.run(
        cache_path=real_arch_cache.facility_points_path(
            amenity_values=real_arch_amenities,
            deduplicate_amenities=real_arch_settings.deduplicate_amenities,
        ),
        builder=lambda: deduplicate_osm_amenities(
            to_point_geometries(
                features,
                projected_epsg=REAL_LUX_CFG.PROJECTED_EPSG,
                verbose=real_arch_settings.verbose,
            ),
            projected_epsg=REAL_LUX_CFG.PROJECTED_EPSG,
            verbose=real_arch_settings.verbose,
        ),
    )


In [ ]:
@derived_artifact(name="real_luxembourg_candidate_sites", schema=PointSchema, format_role="table")
def real_luxembourg_candidate_sites(network_handle: DatasetHandle) -> pd.DataFrame:
    bundle: RealNetworkBundle = network_handle.frame()
    _, candidate_sites_snapped = build_candidate_sites(
        cfg=REAL_LUX_CFG,
        settings=real_arch_settings,
        cache=real_arch_cache,
        nodes=bundle.nodes,
    )
    return candidate_sites_snapped


@derived_artifact(name="real_luxembourg_population_snapped", schema=PointSchema, format_role="table")
def real_luxembourg_population_snapped(population_handle: DatasetHandle, network_handle: DatasetHandle) -> pd.DataFrame:
    bundle: RealNetworkBundle = network_handle.frame()
    return real_arch_cache.run(
        cache_path=real_arch_cache.population_snapped_path_for(
            distance_col="dist_snap_target",
            population_threshold=real_arch_settings.population_threshold,
            sample_fraction=real_arch_settings.sample_fraction,
            max_points=real_arch_settings.max_points,
            aggregate_factor=real_arch_aggregate_factor,
        ),
        builder=lambda: snap_points_to_nodes(
            population_handle.frame(),
            bundle.nodes,
            distance_col="dist_snap_target",
            projected_epsg=REAL_LUX_CFG.PROJECTED_EPSG,
            verbose=real_arch_settings.verbose,
        ),
    )


@derived_artifact(name="real_luxembourg_schools_snapped", schema=PointSchema, format_role="table")
def real_luxembourg_schools_snapped(school_handle: DatasetHandle, network_handle: DatasetHandle) -> pd.DataFrame:
    bundle: RealNetworkBundle = network_handle.frame()
    return real_arch_cache.run(
        cache_path=real_arch_cache.sources_snapped_path_for(
            distance_col="dist_snap_source",
            amenity_values=["source_amenities", *real_arch_source_cache_values],
        ),
        builder=lambda: snap_points_to_nodes(
            school_handle.frame(),
            bundle.nodes,
            distance_col="dist_snap_source",
            projected_epsg=REAL_LUX_CFG.PROJECTED_EPSG,
            verbose=real_arch_settings.verbose,
        ),
    )


In [ ]:
@derived_artifact(name="real_luxembourg_routing_tables", format_role="table_bundle")
def real_luxembourg_routing_tables(
    population_snapped_handle: DatasetHandle,
    schools_snapped_handle: DatasetHandle,
    candidate_sites_handle: DatasetHandle,
) -> dict[str, pd.DataFrame]:
    population = with_prefixed_ids(population_snapped_handle.frame(), "target_population")
    targets = prepare_points_as_targets(population, id_prefix="target")

    source_amenities = with_prefixed_ids(schools_snapped_handle.frame(), "source_amenities")
    source_amenities = prepare_points_as_sources(source_amenities, source_type="existing", id_prefix="source_amenities")

    source_candidates = candidate_layer_for_role(candidate_sites_handle.frame(), role="source", id_prefix="source_candidates")
    source_candidates = prepare_points_as_sources(source_candidates, source_type="candidate", id_prefix="source_candidates")

    sources = concat_layers([source_amenities, source_candidates], label="source")
    existing_sources = concat_layers([source_amenities], label="non-candidate source")
    return {"population": population, "targets": targets, "sources": sources, "existing_sources": existing_sources}


@derived_artifact(name="real_luxembourg_school_distance_matrix", schema=DistanceMatrixSchema, format_role="table")
def real_luxembourg_school_distance_matrix(network_handle: DatasetHandle, routing_tables_handle: DatasetHandle):
    bundle: RealNetworkBundle = network_handle.frame()
    router = RealPandanaRouter().prepare(bundle)
    tables = routing_tables_handle.frame()
    return real_arch_cache.run(
        cache_path=real_arch_cache.distance_matrix_path_for(
            distance_threshold_largest=REAL_LUX_CFG.DISTANCE_THRESHOLD_KM,
            max_total_dist=real_arch_settings.max_total_dist,
            population_threshold=real_arch_settings.population_threshold,
            sample_fraction=real_arch_settings.sample_fraction,
            max_points=real_arch_settings.max_points,
            aggregate_factor=real_arch_aggregate_factor,
            amenity_values=real_arch_source_cache_values,
            candidate_grid_spacing_m=REAL_LUX_CFG.candidate_grid_spacing_m,
            candidate_max_snap_dist_m=REAL_LUX_CFG.candidate_max_snap_dist_m,
            has_candidates=True,
        ),
        builder=lambda: router.route_many(
            targets=tables["targets"],
            sources=tables["sources"],
            threshold_km=REAL_LUX_CFG.DISTANCE_THRESHOLD_KM,
            max_total_dist=real_arch_settings.max_total_dist,
        ),
    )


## Run the Case

In [ ]:
inputs_handle = real_arch_ctx.runner.run(real_luxembourg_inputs)
network_handle = real_arch_ctx.runner.run(real_luxembourg_network)
population_points_handle = real_arch_ctx.runner.run(real_luxembourg_population_points)
school_points_handle = real_arch_ctx.runner.run(real_luxembourg_school_points)
candidate_sites_handle = real_arch_ctx.runner.run(real_luxembourg_candidate_sites, network_handle)
population_snapped_handle = real_arch_ctx.runner.run(real_luxembourg_population_snapped, population_points_handle, network_handle)
schools_snapped_handle = real_arch_ctx.runner.run(real_luxembourg_schools_snapped, school_points_handle, network_handle)
routing_tables_handle = real_arch_ctx.runner.run(
    real_luxembourg_routing_tables,
    population_snapped_handle,
    schools_snapped_handle,
    candidate_sites_handle,
)
real_arch_matrix_handle = real_arch_ctx.runner.run(real_luxembourg_school_distance_matrix, network_handle, routing_tables_handle)

routing_tables = routing_tables_handle.frame()
real_arch_matrix = real_arch_matrix_handle.frame()
matrix_pd = real_arch_matrix.to_pandas() if hasattr(real_arch_matrix, "to_pandas") else real_arch_matrix

summary = {
    "population_points": len(population_points_handle.frame()),
    "school_points": len(school_points_handle.frame()),
    "candidate_sites": len(candidate_sites_handle.frame()),
    "targets": len(routing_tables["targets"]),
    "existing_sources": len(routing_tables["existing_sources"]),
    "sources_total": len(routing_tables["sources"]),
    "distance_matrix_rows": len(matrix_pd),
    "network_handle_reused": network_handle is real_arch_ctx.data.get(network_handle.key),
}
summary


## Figure 1: Population, Schools, and Candidate Sites

In [ ]:
def as_projected_gdf(frame, crs=None):
    if isinstance(frame, gpd.GeoDataFrame):
        result = frame
        if result.crs is None:
            result = result.set_crs(crs or f"EPSG:{REAL_LUX_CFG.PROJECTED_EPSG}")
        return result.to_crs(REAL_LUX_CFG.PROJECTED_EPSG)
    if "geometry" in frame.columns:
        return gpd.GeoDataFrame(
            frame.copy(),
            geometry="geometry",
            crs=crs or f"EPSG:{REAL_LUX_CFG.PROJECTED_EPSG}",
        ).to_crs(REAL_LUX_CFG.PROJECTED_EPSG)
    lon_col = next((c for c in ["Longitude", "longitude", "lon", "x"] if c in frame.columns), None)
    lat_col = next((c for c in ["Latitude", "latitude", "lat", "y"] if c in frame.columns), None)
    if lon_col is None or lat_col is None:
        raise ValueError("Expected either geometry or longitude/latitude columns")
    return gpd.GeoDataFrame(
        frame.copy(),
        geometry=gpd.points_from_xy(frame[lon_col], frame[lat_col]),
        crs="EPSG:4326",
    ).to_crs(REAL_LUX_CFG.PROJECTED_EPSG)

population_gdf = as_projected_gdf(population_points_handle.frame())
schools_gdf = as_projected_gdf(school_points_handle.frame())
candidates_gdf = as_projected_gdf(candidate_sites_handle.frame())

population_plot = population_gdf.sample(min(1500, len(population_gdf)), random_state=7) if len(population_gdf) > 1500 else population_gdf

fig, ax = plt.subplots(figsize=(8, 8))
population_plot.plot(ax=ax, markersize=3, color="#9aa0a6", alpha=0.35, label="Population cells")
candidates_gdf.plot(ax=ax, markersize=30, marker="+", color="#2b8cbe", linewidth=1.0, label="Candidate sites")
schools_gdf.plot(ax=ax, markersize=18, color="#d7301f", alpha=0.85, label="Existing schools")
ax.set_title("Luxembourg school access inputs")
ax.set_axis_off()
ax.legend(loc="lower left", frameon=True)
plt.show()


## Figure 2: Snapping Distances

In [ ]:
pop_snapped = population_snapped_handle.frame()
school_snapped = schools_snapped_handle.frame()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
pop_snapped["dist_snap_target"].hist(ax=axes[0], bins=40, color="#636363")
axes[0].set_title("Population to network snap")
axes[0].set_xlabel("Meters")
axes[0].set_ylabel("Cells")
school_snapped["dist_snap_source"].hist(ax=axes[1], bins=40, color="#d7301f")
axes[1].set_title("Schools to network snap")
axes[1].set_xlabel("Meters")
axes[1].set_ylabel("Schools")
plt.show()


## Figure 3: Retained Distance Matrix

In [ ]:
distance_cols = [c for c in matrix_pd.columns if "dist" in c.lower() or "distance" in c.lower()]
numeric_distance_cols = [c for c in distance_cols if pd.api.types.is_numeric_dtype(matrix_pd[c])]
preferred = ["total_distance", "distance", "distance_m", "network_distance_m"]
distance_col = next((c for c in preferred if c in numeric_distance_cols), numeric_distance_cols[-1] if numeric_distance_cols else None)

fig, ax = plt.subplots(figsize=(8, 4))
if distance_col is not None and len(matrix_pd):
    matrix_pd[distance_col].hist(ax=ax, bins=40, color="#3182bd")
    ax.set_title(f"Retained routes by {distance_col}")
    ax.set_xlabel("Meters")
    ax.set_ylabel("Pairs")
else:
    ax.text(0.5, 0.5, "No numeric distance column found", ha="center", va="center")
    ax.set_axis_off()
plt.show()

{"distance_column_used": distance_col, "matrix_columns": list(matrix_pd.columns)}


## Current Cache and Export Comparison

In [ ]:
LUX_OUTPUTS = REAL_LUX_CFG.BASE_DIR / "outputs"
lux_real_agg10_tag = "pop_1_sample_1_max_none_agg_10_maxdist_1000_amenity_school_candidates_spacing_5000_maxsnap_5000"
exported_matrix_path = LUX_OUTPUTS / f"distance_matrix_{lux_real_agg10_tag}.parquet"
current_matrix_cache_path = real_arch_cache.distance_matrix_path_for(
    distance_threshold_largest=REAL_LUX_CFG.DISTANCE_THRESHOLD_KM,
    max_total_dist=real_arch_settings.max_total_dist,
    population_threshold=real_arch_settings.population_threshold,
    sample_fraction=real_arch_settings.sample_fraction,
    max_points=real_arch_settings.max_points,
    aggregate_factor=real_arch_aggregate_factor,
    amenity_values=real_arch_source_cache_values,
    candidate_grid_spacing_m=REAL_LUX_CFG.candidate_grid_spacing_m,
    candidate_max_snap_dist_m=REAL_LUX_CFG.candidate_max_snap_dist_m,
    has_candidates=True,
)

exported_rows = None
if exported_matrix_path.exists():
    exported_rows = len(pd.read_parquet(exported_matrix_path))

{
    "architecture_rows": len(matrix_pd),
    "current_matrix_cache": current_matrix_cache_path.name,
    "current_matrix_cache_exists": current_matrix_cache_path.exists(),
    "exported_parquet": str(exported_matrix_path),
    "exported_rows": exported_rows,
    "matches_exported_parquet": exported_rows == len(matrix_pd) if exported_rows is not None else None,
}


## Registered Architecture Artifacts

In [ ]:
pd.DataFrame([
    {"artifact": name, "registered_key": str(handle.key), "type": type(handle.frame()).__name__}
    for name, handle in {
        "inputs": inputs_handle,
        "network": network_handle,
        "population_points": population_points_handle,
        "school_points": school_points_handle,
        "candidate_sites": candidate_sites_handle,
        "population_snapped": population_snapped_handle,
        "schools_snapped": schools_snapped_handle,
        "routing_tables": routing_tables_handle,
        "distance_matrix": real_arch_matrix_handle,
    }.items()
])
